In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import asyncio

import httpx
import json

from a2a.client import ClientConfig, create_client
from a2a.client.card_resolver import A2ACardResolver
from a2a.helpers import get_artifact_text, get_message_text, new_text_message
from a2a.types import (
    Message,
    Part,
    Role,
    SendMessageRequest,
    SubscribeToTaskRequest,
    GetTaskRequest,
    ListTasksRequest,
    TaskState,
)
from a2a.utils import TransportProtocol

In [3]:
from pydantic import BaseModel, Field

In [4]:
from typing import List, Union, Optional, Literal, Dict

In [5]:
from langchain_core.tools import tool

In [6]:
from IPython.display import display, Markdown

In [7]:
from agent.agent import Agent
from agent.thread import Thread
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    base_url= os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
    model=os.getenv("LLM"),
)

In [8]:
class A2ARequest(BaseModel):
    url: str = Field(..., description="Request url")
    protocol: Literal[TransportProtocol.JSONRPC, TransportProtocol.HTTP_JSON] = Field(..., description="Transport Protocol")
    text: str = Field(..., description="Query/Message/Text to send to the agent")

In [11]:
@tool("a2a_invoke")
async def a2a_invoke(
   request: A2ARequest 
):
    """
    Use it to assign task/work to an another agent using the A2A Protocol
    """
    async with httpx.AsyncClient(timeout=None) as http:
        agent_card = await A2ACardResolver(http, request.url).get_agent_card()
        
        allowed_protocols = [p.protocol_binding for p in card.supported_interfaces]
        client = await create_client(
            agent_card,
            client_config = ClientConfig(
                supported_protocol_bindings = [
                    request.protocol 
                    if request.protocol in allowed_protocols
                    else allowed_protocols[0]
                ],
                httpx_client = http,
            ),
        )
    
        try:
            request = SendMessageRequest(
                message=new_text_message(
                    text=request.text, 
                    role=Role.ROLE_USER,
                ),
            )
            async for reply in client.send_message(request):
                response = reply
                break
        finally:
            await client.close()

    return json.dumps(
        {
            "task_id": response.task.id,
            "task_context_id": response.task.context_id,
            "terminate_state": TaskState.Name(response.task.status.state),
            "text": response.task.status.message.parts[0].text,
        }
    )

In [12]:
agent_card_urls = ["http://localhost:8002"]

In [13]:
agent_cards = []
async with httpx.AsyncClient() as http:
    for url in agent_card_urls:
        agent_cards.append(await A2ACardResolver(http, url).get_agent_card())

In [14]:
agent_cards_info = """
============
AGENT CARDS
============
"""

for card in agent_cards:
    agent_cards_info += f"\n{card}\n---"

In [15]:
agent = Agent(
    model=llm,
    tools=[a2a_invoke]
)

In [16]:
thread = Thread()

In [17]:
system = f"""
You are an orchestrator and helper agent. You will be provided an `AGENT CARDS`. You can call/assign task to agents in the `AGENT CARDS` as per required or needed

{agent_cards_info}
"""

In [18]:
SystemMessage(system) | thread

In [19]:
HumanMessage("list all main 5 types sql injection, just need names, use and try the a2a invoke tool once if working otherwise return the issue of tool calling") | thread

In [20]:
response = agent.invoke(thread)

In [21]:
display(Markdown(response.content))

The A2A invoke tool is working correctly. The CyberSecurity Agent successfully returned the response.

Here are the main 5 types of SQL injection:

1. **Error-based SQL Injection**
2. **Union-based SQL Injection**
3. **Boolean-based Blind SQL Injection**
4. **Time-based Blind SQL Injection**
5. **Out-of-band SQL Injection**

The tool call was successful with no issues encountered.

In [22]:
thread

{"depth": 0, "system": 1, "human": 1, "ai": 2, "tool": 1}

In [23]:
for t in thread:
    print(t.cont)
    print("\n<==============>\n")

content='\nYou are an orchestrator and helper agent. You will be provided an `AGENT CARDS`. You can call/assign task to agents in the `AGENT CARDS` as per required or needed\n\n\n============\nAGENT CARDS\n============\n\nname: "CyberSecurity Agent"\ndescription: "It will answer to any cybersecurity related query"\nsupported_interfaces {\n  url: "http://localhost:8002/"\n  protocol_binding: "JSONRPC"\n}\nversion: "0.1.0"\ncapabilities {\n  streaming: false\n  push_notifications: false\n}\ndefault_input_modes: "text/plain"\ndefault_output_modes: "text/plain"\nskills {\n  id: "cybersecurity_analysis"\n  name: "Cybersecurity Analysis"\n  description: "Analyzes and answers cybersecurity-related questions, including vulnerability assessment, penetration testing, security concepts, threat analysis, and security best practices."\n  tags: "cybersecurity"\n  tags: "vulnerability-assessment"\n  tags: "penetration-testing"\n  tags: "security-analysis"\n  tags: "threat-analysis"\n  examples: "Expl

In [13]:
text = "tell me about Blind SQL Injection"

In [14]:
async with httpx.AsyncClient(timeout=None) as http:
    client = await create_client(
        card,
        client_config = ClientConfig(
            supported_protocol_bindings = [
                card.supported_interfaces[0].protocol_binding
                # TransportProtocol.JSONRPC
            ],
            httpx_client = http,
        ),
    )

    try:

        request = SendMessageRequest(
            message=new_text_message(
                text=text, 
                role=Role.ROLE_USER,
            ),
        )
        async for reply in client.send_message(request):
            print(reply)
            response = reply
            break
    finally:
        await client.close()

task {
  id: "e1b75c4f-5db4-487d-9b53-685300caaf43"
  context_id: "4a10172a-64fe-4744-b430-0a2a3d0e88c5"
  status {
    state: TASK_STATE_COMPLETED
    message {
      message_id: "d81650c7-81b2-4d15-a7fb-4753bb8feb95"
      context_id: "4a10172a-64fe-4744-b430-0a2a3d0e88c5"
      task_id: "e1b75c4f-5db4-487d-9b53-685300caaf43"
      role: ROLE_AGENT
      parts {
        text: "## Blind SQL Injection: An In-Depth Analysis\n\n### 1. Definition and Core Concept\n\nBlind SQL Injection is a sophisticated attack technique used against web applications that are vulnerable to SQL injection but do not return the results of a SQL query directly in the application’s response. Unlike classic SQL injection, where error messages or data directly appear in the output, blind SQLi requires the attacker to infer information by observing the application’s behavior (e.g., HTTP response status, response time, or content changes).\n\nThe term “blind” refers to the fact that the attacker cannot see the dat

In [15]:
print(response)

task {
  id: "e1b75c4f-5db4-487d-9b53-685300caaf43"
  context_id: "4a10172a-64fe-4744-b430-0a2a3d0e88c5"
  status {
    state: TASK_STATE_COMPLETED
    message {
      message_id: "d81650c7-81b2-4d15-a7fb-4753bb8feb95"
      context_id: "4a10172a-64fe-4744-b430-0a2a3d0e88c5"
      task_id: "e1b75c4f-5db4-487d-9b53-685300caaf43"
      role: ROLE_AGENT
      parts {
        text: "## Blind SQL Injection: An In-Depth Analysis\n\n### 1. Definition and Core Concept\n\nBlind SQL Injection is a sophisticated attack technique used against web applications that are vulnerable to SQL injection but do not return the results of a SQL query directly in the application’s response. Unlike classic SQL injection, where error messages or data directly appear in the output, blind SQLi requires the attacker to infer information by observing the application’s behavior (e.g., HTTP response status, response time, or content changes).\n\nThe term “blind” refers to the fact that the attacker cannot see the dat

In [16]:
async with httpx.AsyncClient(timeout=None) as http:
    client = await create_client(
        card,
        client_config = ClientConfig(
            supported_protocol_bindings = [
                card.supported_interfaces[0].protocol_binding
                # TransportProtocol.JSONRPC
            ],
            httpx_client = http,
        ),
    )

    try:

        # request = SendMessageRequest(
        #     message=new_text_message(
        #         text=text, 
        #         role=Role.ROLE_USER,
        #     ),
        # )
        # async for reply in client.send_message(GetTaskRequest()):
        #     response = reply
        #     break
        # request = GetTaskRequest(
        #     id=response.task.id
        # )

        # task = await client.get_task(request)

        # print("Task ID:", task.id)
        # print("Context ID:", task.context_id)
        # print("Status:", task.status.state)
        # print("Artifacts:", task.artifacts)
        response = await client.list_tasks(ListTasksRequest())
    finally:
        await client.close()

In [22]:
response.tasks[1]

id: "83b292a9-b9e1-4896-9364-de851fe1487b"
context_id: "bf866a42-688e-4f0a-8793-2e74f2ab8537"
status {
  state: TASK_STATE_COMPLETED
  message {
    message_id: "ac4e8112-c78b-4039-8f9f-832847af83e5"
    context_id: "bf866a42-688e-4f0a-8793-2e74f2ab8537"
    task_id: "83b292a9-b9e1-4896-9364-de851fe1487b"
    role: ROLE_AGENT
    parts {
      text: "I can\'t answer your query. Please ask a Cybersecurity related query."
    }
  }
  timestamp {
    seconds: 1789536409
    nanos: 831389000
  }
}
history {
  message_id: "141925bd-83da-4b67-a8ef-de9225f08cf2"
  context_id: "bf866a42-688e-4f0a-8793-2e74f2ab8537"
  task_id: "83b292a9-b9e1-4896-9364-de851fe1487b"
  role: ROLE_USER
  parts {
    text: "tell me about Narendra Modi"
  }
}
history {
  message_id: "f7125613-d87c-4055-9c0f-a480f26610f7"
  context_id: "bf866a42-688e-4f0a-8793-2e74f2ab8537"
  task_id: "83b292a9-b9e1-4896-9364-de851fe1487b"
  role: ROLE_AGENT
  parts {
    text: "Working on the task."
  }
}

In [12]:
task

id: "0fe8cc38-2745-4fda-9faf-fc58ed87c4e4"
context_id: "184b7bb9-f78e-4577-bc64-11f2013a1e97"
status {
  state: TASK_STATE_COMPLETED
  message {
    message_id: "b660731b-6916-477a-b913-d7cffae10151"
    context_id: "184b7bb9-f78e-4577-bc64-11f2013a1e97"
    task_id: "0fe8cc38-2745-4fda-9faf-fc58ed87c4e4"
    role: ROLE_AGENT
    parts {
      text: "I can\'t answer your query. Please ask Cybersecurity related query."
    }
  }
  timestamp {
    seconds: 1789534097
    nanos: 646071000
  }
}
history {
  message_id: "5ad8b666-7b25-4eec-adb1-297e539231fd"
  context_id: "184b7bb9-f78e-4577-bc64-11f2013a1e97"
  task_id: "0fe8cc38-2745-4fda-9faf-fc58ed87c4e4"
  role: ROLE_USER
  parts {
    text: "Tell me about Nike"
  }
}
history {
  message_id: "2e17f76d-63ef-4a71-b4dd-9655d1423fd4"
  context_id: "184b7bb9-f78e-4577-bc64-11f2013a1e97"
  task_id: "0fe8cc38-2745-4fda-9faf-fc58ed87c4e4"
  role: ROLE_AGENT
  parts {
    text: "Working on the task."
  }
}

In [12]:
display(Markdown(response.task.status.message.parts[0].text))

# Blind SQL Injection: In-Depth Report

## 1. Introduction

Blind SQL Injection is a subtype of SQL Injection attack where an attacker can extract data from a database by sending SQL queries that cause the application to behave differently (e.g., in response times or returned content) without directly displaying the database output. Unlike classic SQL Injection (error‑based or UNION‑based), blind injection does not rely on error messages or visible data from the database; instead, it exploits subtle changes in the application’s response to infer information bit by bit.

## 2. How Blind SQL Injection Works

Blind SQL Injection typically occurs when an application is vulnerable to SQL injection but the database output is not directly returned to the user. The attacker must ask the database a series of yes/no questions (or true/false conditions) and observe the application’s behavior to deduce the answer.

Two primary techniques exist:

### 2.1 Boolean-Based Blind SQL Injection
The attacker sends a SQL query with a condition that evaluates to either `TRUE` or `FALSE`. The application’s response differs between the two states (e.g., one page returns content, another returns a different page or an error). By observing these differences, the attacker can infer whether a condition is true.

**Example (in URL parameter):**
```
http://example.com/product?id=1 AND 1=1   -- True → page loads normally
http://example.com/product?id=1 AND 1=2   -- False → page may be empty or show error
```

The attacker can then progressively extract data:
- `id=1 AND (SELECT SUBSTRING(password,1,1) FROM users WHERE username='admin') = 'a'` → True/False

### 2.2 Time-Based Blind SQL Injection
If the application shows no visible difference between TRUE and FALSE conditions, the attacker introduces a time delay (e.g., using `SLEEP()` or `WAITFOR DELAY`) to create a measurable difference. If a condition is true, the server pauses; if false, it responds immediately.

**Example (MySQL):**
```
id=1 AND IF(SUBSTRING(password,1,1)='a', SLEEP(5), 0)
```
By measuring response time, the attacker can brute‑force characters.

## 3. Detailed Attack Workflow

1. **Identify the injection point** – Usually a parameter (GET/POST) that is directly concatenated into an SQL query without sanitization.
2. **Confirm blind vulnerability** – Send a TRUE condition (e.g., `1=1`) and a FALSE condition (`1=2`) and observe a consistent difference.
3. **Extract database metadata** – Determine the database version, table names, column names, etc., using Boolean logic and substring extraction.
4. **Extract data** – Brute‑force character by character. For each position, test all possible characters (e.g., 0–9, a–z, special characters) until the condition returns TRUE.
5. **Automation** – Tools like `sqlmap` automate the entire process and can extract the entire database schema and content.

### 3.1 Example of Manual Boolean Extraction
Assuming an application returns “Product found” when `id=1 AND 1=1` and “Not found” when `id=1 AND 1=2`.

To get the first character of the database name:
```
id=1 AND SUBSTRING(database(),1,1)='a'   → returns "Not found"
id=1 AND SUBSTRING(database(),1,1)='b'   → returns "Not found"
...
id=1 AND SUBSTRING(database(),1,1)='m'   → returns "Product found" (MySQL databases often start with 'm')
```

Each correct guess takes one request. For a 10‑character database name, an average of 1280 requests (10 × 128 ASCII possibilities / 2) would be needed in the worst case.

## 4. Common Database Functions Used

- **Substring extraction:** `SUBSTRING(string, start, length)` (MySQL), `SUBSTR()` (Oracle), `SUBSTRING()` (SQL Server)
- **ASCII/char conversion:** `ASCII('a')` or `CHAR(97)` to compare numeric values (reduces character set to numbers 32–126)
- **Conditional delays:**
  - MySQL: `SLEEP(seconds)`
  - SQL Server: `WAITFOR DELAY '0:0:5'`
  - PostgreSQL: `pg_sleep(seconds)`
  - Oracle: `DBMS_LOCK.SLEEP(seconds)` (requires privileges) or heavy queries (e.g., `SELECT COUNT(*) FROM all_objects`)

## 5. Detection & Prevention

### 5.1 Detection
- **Dynamic Application Security Testing (DAST)** tools send payloads and monitor response differences.
- **Static Code Analysis** to find unsanitized user input concatenated in SQL queries.
- **Log monitoring** – Look for abnormal percentages of TRUE/FALSE requests or high response times for a single parameter.

### 5.2 Prevention
1. **Use parameterized queries (prepared statements)** – Never concatenate user input directly into SQL.
2. **Input validation** – Whitelist acceptable values (e.g., integers for IDs).
3. **Least privilege principle** – Database user should have minimal permissions; restrict access to `information_schema` and system tables.
4. **Web Application Firewall (WAF)** – Can block known blind SQL injection patterns (e.g., `SLEEP`, `OR 1=1`).
5. **Disable error messages** – Prevent attackers from seeing database errors.
6. **Rate limiting** – Protect against brute‑force enumeration.

## 6. Impact

| Impact Area | Consequences |
|-------------|--------------|
| **Data breach** | Extraction of passwords, credit cards, personal data. |
| **Authentication bypass** | Login forms with blind injection can be bypassed without returning data. |
| **Database corruption** | Blind injection can be used to run `UPDATE` or `DELETE` statements (if the database user has write privileges). |
| **Privilege escalation** | Extract admin credentials or modify user roles. |
| **Denial of Service** | Time‑based injection can overload the database with delays. |

## 7. Real-World Example (CVE‑2023‑XXXX)

Many legacy applications (e.g., older PHP + MySQL apps) suffer from blind injection in search fields. For instance, an e‑commerce site with a product search like:
```php
$query = "SELECT * FROM products WHERE name LIKE '%".$_GET['search']."%'";
```
An attacker could use boolean‑based injection to enumerate product inventory or extract customer email addresses from a related database table.

## 8. Conclusion

Blind SQL Injection is a stealthy but powerful attack vector. While slower than in‑band injection, it allows an attacker to fully reconstruct a database without any visible output. Defenders must treat every user input as untrusted and enforce parameterized queries, combined with solid security practices such as least privilege, input validation, and robust monitoring.

For further reading, study the OWASP Guide on SQL Injection or practice in a controlled lab environment (e.g., TryHackMe, PortSwigger Web Security Academy).